In [25]:
import re, io
from pathlib import Path
import json
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\223114186\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\223114186\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [5]:
DATA_DIR = Path("C:/Users/223114186/Desktop/TP3_Word Embeddings/data")
TEXT_PATH = DATA_DIR / 'text8.text'

In [6]:
raw = Path(TEXT_PATH).read_text(encoding='utf-8', errors='ignore')
# Normalisation simple
text = raw.lower()
text = re.sub(r"[^a-z\s]", " ", text)
text = re.sub(r"\s+", " ", text).strip()


# Tokenisation
tokens = nltk.word_tokenize(text)


# Stopwords anglais (optionnel selon objectif)
stops = set(stopwords.words('english'))
# On peut *ne pas* retirer les stops pour Word2Vec; ici on laisse une variable
keep_stops = False
if not keep_stops:
    tokens = [t for t in tokens if t not in stops]


print('Nb tokens:', len(tokens))

Nb tokens: 2424445


In [17]:
from collections import Counter
min_freq = 5
freq = Counter(tokens)
vocab = [w for w,c in freq.items() if c >= min_freq]
vocab = sorted(vocab)
word2id = {w:i for i,w in enumerate(vocab)}
id2word = {i:w for w,i in word2id.items()}


unk_id = len(vocab)


ids = [word2id.get(w, unk_id) for w in tokens]
print('Vocab size:', len(vocab)+1, '(+UNK)')

Vocab size: 31168 (+UNK)


In [13]:
print("Nombre total de tokens :", len(tokens))
print("Exemple de 20 tokens :", tokens[:20])
print("Taille du vocabulaire (min_freq=5):", len(vocab))
print("Exemples vocab :", list(vocab)[:20])

Nombre total de tokens : 2424445
Exemple de 20 tokens : ['anarchism', 'originated', 'term', 'abuse', 'first', 'used', 'early', 'working', 'class', 'radicals', 'including', 'diggers', 'english', 'revolution', 'sans', 'culottes', 'french', 'revolution', 'whilst', 'term']
Taille du vocabulaire (min_freq=5): 31167
Exemples vocab : ['aa', 'aaa', 'aaas', 'aac', 'aachen', 'aage', 'aalborg', 'aalen', 'aaliyah', 'aalto', 'aam', 'aar', 'aarau', 'aardvark', 'aardvarks', 'aardwolf', 'aargau', 'aarhus', 'aaron', 'aarseth']


In [23]:

# ids = [word2id.get(w, unk_id) for w in tokens]

ids_arr = np.asarray(ids, dtype=np.int32)
N = ids_arr.shape[0]
w = 2  # taille demi-fenêtre

assert N > 2*w, (N, w)

# Cible (mot au centre)
y_tgt = ids_arr[w: N-w]

# Contexte: concatène les w mots de gauche puis les w de droite
context_slices = []
# gauche: offsets négatifs
for off in range(-w, 0):
    context_slices.append(ids_arr[w + off : N - w + off])
# droite: offsets positifs
for off in range(1, w+1):
    context_slices.append(ids_arr[w + off : N - w + off])

# Empile en colonnes → (N-2w, 2w)
X_ctx = np.column_stack(context_slices).astype(np.int32)

print("Shapes:", X_ctx.shape, y_tgt.shape)  # attendu ~ (N-2w, 2w) et (N-2w,)


Shapes: (2424441, 4) (2424441,)


In [26]:
Path_artifact = Path("C:/Users/223114186/Desktop/TP3_Word Embeddings/artifact")

# Sauvegarde des variables nécessaires pour les modèles suivants
np.savez(Path_artifact / "text8_dataset.npz",
         X_ctx=X_ctx,
         y_tgt=y_tgt,
         unk_id=unk_id)

# Sauvegarde du vocabulaire et des mappings
with open(Path_artifact / "vocab_mappings.json", "w", encoding="utf-8") as f:
    json.dump({
        "vocab": vocab,
        "word2id": word2id,
        "id2word": id2word
    }, f)

print("✅ Sauvegarde terminée : X_ctx, y_tgt, vocab, word2id/id2word")

✅ Sauvegarde terminée : X_ctx, y_tgt, vocab, word2id/id2word
